# 🎤 RVC-Python: Voice Conversion Demo

This notebook demonstrates **RVC (Retrieval-based Voice Conversion)** using the `rvc-python` library.

### What you can do:
- Download RVC base models and custom voice models from **any link**
- Upload your own audio and a pre-trained RVC model
- Convert voices with different pitch extraction methods
- Adjust pitch, protection, and other parameters via **interactive forms**
- Run everything on a free Google Colab GPU
- Uses **uv** for blazing fast dependency installation

### Supported download sources:
- 🔗 **Direct URL** (`.pth`, `.index`, `.pt`)
- 🤗 **HuggingFace** (auto-converts to raw download URL)
- 📁 **Google Drive** (file sharing links & folders)
- 📦 **Any public link** (Mega, Dropbox, etc.)

---

## 1️⃣ Check GPU & Install Dependencies

Make sure you're using a **GPU runtime** (Runtime > Change runtime type > T4 GPU).

Uses [uv](https://github.com/astral-sh/uv) for 10-100x faster package installation.

In [ ]:
!nvidia-smi -L

In [ ]:
# @title ⚡ Fast Install with uv
# @markdown Install [uv](https://docs.astral.sh/uv/) for ultra-fast pip operations.

# @markdown ---
CUDA_VERSION = "cu121"  # @param ["cu121", "cu124", "cu126", "cpu"]
INSTALL_RVC_FROM = "git+https://github.com/onxlmao/rvc-python.git"  # @param {type:"string", description:"Custom rvc-python install source (git URL or PyPI package)"}
# @markdown ---
INSTALL_DOWNLOAD_HELPERS = True  # @param {type:"boolean", description:"Install gdown for Google Drive support"}

import subprocess, sys, os

# Step 1: Install uv ( blazing fast Python package manager)
print("⚡ Installing uv...")
subprocess.run([sys.executable, "-m", "pip", "install", "uv", "-q"], check=True)

# Step 2: Use uv to install PyTorch
print(f"📦 Installing PyTorch ({CUDA_VERSION}) via uv...")
index_url = {
    "cu121": "https://download.pytorch.org/whl/cu121",
    "cu124": "https://download.pytorch.org/whl/cu124",
    "cu126": "https://download.pytorch.org/whl/cu126",
    "cpu":  "https://download.pytorch.org/whl/cpu",
}[CUDA_VERSION]
subprocess.run([
    "uv", "pip", "install",
    "torch", "torchvision", "torchaudio",
    "--index-url", index_url,
    "-q"
], check=True)

# Step 3: Install rvc-python
print(f"📦 Installing rvc-python via uv...")
subprocess.run(["uv", "pip", "install", INSTALL_RVC_FROM, "-q"], check=True)

# Step 4: Optional helpers
if INSTALL_DOWNLOAD_HELPERS:
    print("📦 Installing download helpers (gdown) via uv...")
    subprocess.run(["uv", "pip", "install", "gdown", "-q"], check=True)

print("\n✅ All dependencies installed via uv!")

In [ ]:
# @title ✅ Verify Installation
import torch

print("=" * 40)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM:            {mem:.1f} GB")

# Verify rvc-python
try:
    import rvc_python
    print(f"rvc-python:      ✅ installed")
except ImportError:
    print(f"rvc-python:      ❌ not found")

print("=" * 40)

## 2️⃣ Download Models

Use the forms below to download RVC models from **any link**. The notebook auto-detects the source type.

| Source | Example |
|--------|----------|
| **Direct URL** | `https://example.com/model.pth` |
| **HuggingFace** | `https://huggingface.co/user/repo/blob/main/model.pth` |
| **Google Drive** | `https://drive.google.com/file/d/FILE_ID/view` |

In [ ]:
# @title 📁 Setup Directories
import os

os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("base_model", exist_ok=True)

for d in ["input", "output", "models", "base_model"]:
    print(f"  ✅ {d}/")

### 📦 Download Base Models (Required)

These base models are **required** for RVC inference (HuBERT + RMVPE).

In [ ]:
# @title Download RVC Base Models
# @markdown Download HuBERT + RMVPE base models required for inference.

BASE_MODEL_DIR = "base_model"

base_files = {
    "hubert_base.pt": "https://huggingface.co/Daswer123/RVC_Base/resolve/main/hubert_base.pt",
    "rmvpe.pt":       "https://huggingface.co/Daswer123/RVC_Base/resolve/main/rmvpe.pt",
    "rmvpe.onnx":     "https://huggingface.co/Daswer123/RVC_Base/resolve/main/rmvpe.onnx",
}

import requests

print("📦 Downloading RVC base models...")
print("=" * 50)

for filename, url in base_files.items():
    filepath = os.path.join(BASE_MODEL_DIR, filename)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"✅ {filename} already exists ({size_mb:.1f} MB)")
        continue
    
    print(f"⬇️  Downloading {filename}...")
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(filepath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total > 0:
                    pct = downloaded / total * 100
                    print(f"\r   Progress: {pct:.0f}% ({downloaded/1024/1024:.1f}/{total/1024/1024:.1f} MB)", end="")
        print(f"\n   ✅ Saved to {filepath}")
    except Exception as e:
        print(f"\n   ❌ Failed to download {filename}: {e}")

print("\n" + "=" * 50)
print("✅ Base model download complete!")

### 🎤 Download Custom Voice Model from Any Link

Paste **any link** in the form below — it auto-detects the source type.

- **Direct URL**: ends with `.pth`, `.index`, `.zip`
- **HuggingFace**: any HF URL (auto-converted to raw download)
- **Google Drive**: `/file/d/` or `/drive/folders/` links
- **ZIP archives**: auto-extracted to `models/`

In [ ]:
# @title 📥 Download Model from Any Link
import re, os, zipfile, tarfile, requests, shutil
from urllib.parse import urlparse

# @markdown ---
MODEL_LINK = ""  # @param {type:"string", description:"Paste your model link here (HuggingFace, Google Drive, direct URL...)"}
MODEL_NAME = ""  # @param {type:"string", description:"Custom filename (optional, auto-detected if empty)"}
# @markdown ---

def detect_source(url):
    url_lower = url.lower()
    if "drive.google.com/file/d/" in url_lower:
        return "gdrive_file"
    if "drive.google.com/drive/folders/" in url_lower:
        return "gdrive_folder"
    if "huggingface.co" in url_lower:
        return "huggingface"
    return "direct"

def extract_gdrive_id(url):
    m = re.search(r'/file/d/([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1), "file"
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1), "folder"
    m = re.search(r'open\?id=([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1), "file"
    return None, None

def convert_hf_to_download_url(url):
    url_converted = re.sub(r'/blob/', '/resolve/', url)
    if '/tree/' in url_converted:
        return url_converted, "tree"
    return url_converted, "file"

def guess_filename_from_url(url, source_type):
    if source_type == "huggingface":
        m = re.search(r'/resolve/main/(.+)', url)
        if m:
            return m.group(1).split('/')[-1]
    parsed = urlparse(url)
    basename = os.path.basename(parsed.path)
    if basename and '.' in basename:
        return basename
    return None

def extract_archive(filepath, dest_dir):
    print(f"📦 Extracting archive...")
    if filepath.endswith('.zip'):
        with zipfile.ZipFile(filepath, 'r') as zf:
            zf.extractall(dest_dir)
    elif filepath.endswith(('.tar.gz', '.tgz')):
        with tarfile.open(filepath, 'r:gz') as tf:
            tf.extractall(dest_dir)
    elif filepath.endswith('.tar'):
        with tarfile.open(filepath, 'r') as tf:
            tf.extractall(dest_dir)
    else:
        return False
    return True

def download_with_progress(url, filepath):
    r = requests.get(url, stream=True, timeout=120, allow_redirects=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(filepath, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                pct = downloaded / total * 100
                bar_len = 30
                filled = int(bar_len * pct / 100)
                bar = '█' * filled + '░' * (bar_len - filled)
                print(f"\r   [{bar}] {pct:.0f}% ({downloaded/1024/1024:.1f}/{total/1024/1024:.1f} MB)", end="")
    print()
    return filepath

def find_model_files(directory):
    pth_files, index_files = [], []
    for root, dirs, files in os.walk(directory):
        for f in files:
            full = os.path.join(root, f)
            if f.endswith('.pth'):
                pth_files.append(full)
            elif f.endswith('.index'):
                index_files.append(full)
    return pth_files, index_files

def process_downloaded(filepath):
    """Extract if archive, find model/index files."""
    global MODEL_PATH, INDEX_PATH
    if filepath.endswith(('.zip', '.tar.gz', '.tgz', '.tar')):
        extract_archive(filepath, "models")
        os.remove(filepath)
        pth_files, idx_files = find_model_files("models")
        if pth_files:
            MODEL_PATH = pth_files[0]
            print(f"🎯 Model found: {MODEL_PATH}")
        if idx_files:
            INDEX_PATH = idx_files[0]
            print(f"🎯 Index found: {INDEX_PATH}")
<arg_value>    else:
        if filepath.endswith('.index'):
            INDEX_PATH = filepath
            print(f"🎯 Index saved: {INDEX_PATH}")
        elif filepath.endswith('.pth'):
            MODEL_PATH = filepath
        else:
            print(f"⚠️  Downloaded file is not .pth or .index: {filepath}")
            MODEL_PATH = filepath  # try anyway

# ====== MAIN DOWNLOAD LOGIC ======

if not MODEL_LINK.strip():
    print("⚠️  No link provided. Skipping download.")
    print("   You can upload models manually in the next section.")
    MODEL_PATH = ""
    INDEX_PATH = ""
else:
    source = detect_source(MODEL_LINK.strip())
    print(f"🔍 Detected source: {source}")
    print(f"🔗 Link: {MODEL_LINK.strip()}")
    print()
    
    MODEL_PATH = ""
    INDEX_PATH = ""
    
    if source == "gdrive_file":
        file_id, _ = extract_gdrive_id(MODEL_LINK.strip())
        if file_id:
            gdrive_url = f"https://drive.google.com/uc?export=download&id={file_id}&confirm=t"
            filename = MODEL_NAME if MODEL_NAME else "gdrive_model.pth"
            filepath = os.path.join("models", filename)
            print(f"⬇️  Downloading from Google Drive...")
            try:
                download_with_progress(gdrive_url, filepath)
                print(f"✅ Saved to {filepath}")
                process_downloaded(filepath)
            except Exception as e:
                print(f"❌ Google Drive download failed: {e}")
                print("   Trying with gdown fallback...")
                try:
                    import gdown
                    output = gdown.download(f"https://drive.google.com/uc?id={file_id}", "models/", quiet=False, fuzzy=True)
                    if output:
                        MODEL_PATH = output
                        print(f"✅ Downloaded via gdown: {output}")
                except Exception as e2:
                    print(f"❌ gdown also failed: {e2}")
    
    elif source == "gdrive_folder":
        folder_id, _ = extract_gdrive_id(MODEL_LINK.strip())
        if folder_id:
            print(f"⬇️  Downloading Google Drive folder...")
            try:
                import gdown
                output_dir = gdown.download_folder(f"https://drive.google.com/drive/folders/{folder_id}", output="models/", quiet=False)
                if output_dir:
                    print(f"✅ Folder downloaded to models/")
                    pth_files, idx_files = find_model_files("models")
                    if pth_files:
                        MODEL_PATH = pth_files[0]
                        print(f"🎯 Model: {MODEL_PATH}")
                    if idx_files:
                        INDEX_PATH = idx_files[0]
                        print(f"🎯 Index: {INDEX_PATH}")
                else:
                    print("❌ Failed to download folder.")
            except ImportError:
                print("❌ gdown not installed. Enable 'Install download helpers' in step 1.")
            except Exception as e:
                print(f"❌ Folder download failed: {e}")
    
    elif source == "huggingface":
        download_url, hf_type = convert_hf_to_download_url(MODEL_LINK.strip())
        
        if hf_type == "tree":
            print("📂 HuggingFace repo detected. Scanning for model files...")
            try:
                repo_match = re.match(r'https://huggingface\.co/([^/]+)/([^/]+)', MODEL_LINK.strip())
                if repo_match:
                    org, repo = repo_match.group(1), repo_match.group(2)
                    api_url = f"https://huggingface.co/api/models/{org}/{repo}"
                    resp = requests.get(api_url, timeout=30)
                    if resp.status_code == 200:
                        siblings = resp.json().get('siblings', [])
                        downloadable = [s['rfilename'] for s in siblings if not s['rfilename'].endswith('/')]
                        print(f"   Found {len(downloadable)} files:")
                        for f in downloadable:
                            print(f"     - {f}")
                        
                        for fname in downloadable:
                            if fname.endswith(('.pth', '.index', '.pt', '.onnx')):
                                furl = f"https://huggingface.co/{org}/{repo}/resolve/main/{fname}"
                                fpath = os.path.join("models", fname)
                                if not os.path.exists(fpath):
                                    print(f"\n⬇️  {fname}...")
                                    try:
                                        download_with_progress(furl, fpath)
                                        print(f"   ✅ Saved")
                                    except Exception as e:
                                        print(f"   ❌ {e}")
                                else:
                                    print(f"   ✅ {fname} already exists")
                        
                        pth_files, idx_files = find_model_files("models")
                        if pth_files:
                            MODEL_PATH = pth_files[0]
                            print(f"\n🎯 Model: {MODEL_PATH}")
                        if idx_files:
                            INDEX_PATH = idx_files[0]
                            print(f"🎯 Index: {INDEX_PATH}")
                    else:
                        print(f"❌ Could not list repo (status {resp.status_code})")
            except Exception as e:
                print(f"❌ HuggingFace scan failed: {e}")
        else:
            guessed_name = guess_filename_from_url(download_url, "huggingface")
            filename = MODEL_NAME if MODEL_NAME else (guessed_name or "hf_model.pth")
            filepath = os.path.join("models", filename)
            
            print(f"⬇️  Downloading from HuggingFace...")
            print(f"   URL: {download_url}")
            try:
                download_with_progress(download_url, filepath)
                print(f"✅ Saved to {filepath}")
                process_downloaded(filepath)
            except Exception as e:
                print(f"❌ HuggingFace download failed: {e}")
    
    else:  # direct URL
        guessed_name = guess_filename_from_url(MODEL_LINK.strip(), "direct")
        filename = MODEL_NAME if MODEL_NAME else (guessed_name or "downloaded_model.pth")
        filepath = os.path.join("models", filename)
        
        print(f"⬇️  Downloading from direct URL...")
        try:
            download_with_progress(MODEL_LINK.strip(), filepath)
            print(f"✅ Saved to {filepath}")
            process_downloaded(filepath)
        except Exception as e:
            print(f"❌ Direct download failed: {e}")

print("\n" + "=" * 50)
print("📋 Summary:")
print(f"   Model (.pth):  {MODEL_PATH if MODEL_PATH else '(not set)'}")
print(f"   Index (.index): {INDEX_PATH if INDEX_PATH else '(not set)'}")
print("=" * 50)

In [ ]:
# @title 📥 Download Index File (Optional)
# @markdown Paste your .index file link (leave empty to skip).
INDEX_LINK = ""  # @param {type:"string", description:"Paste .index file link here (optional)"}

if INDEX_LINK.strip() and not INDEX_PATH:
    source = detect_source(INDEX_LINK.strip())
    print(f"🔍 Source: {source}")
    
    if source == "huggingface":
        download_url, _ = convert_hf_to_download_url(INDEX_LINK.strip())
    else:
        download_url = INDEX_LINK.strip()
    
    guessed_name = guess_filename_from_url(download_url, source)
    filename = guessed_name if guessed_name else "added_index.index"
    filepath = os.path.join("models", filename)
    
    print(f"⬇️  Downloading index file...")
    try:
        download_with_progress(download_url, filepath)
        INDEX_PATH = filepath
        print(f"✅ Index saved to {INDEX_PATH}")
    except Exception as e:
        print(f"❌ Download failed: {e}")
elif INDEX_PATH:
    print(f"✅ Index already set: {INDEX_PATH}")
else:
    print("⏭️  No index link provided. Index will be empty.")

print(f"\n📌 Index path: {INDEX_PATH if INDEX_PATH else '(none)'}")

In [ ]:
# @title 📂 List All Model Files
import os

print("📂 models/")
print("-" * 40)
if os.path.exists("models"):
    items = os.listdir("models")
    if items:
        for f in sorted(items):
            size = os.path.getsize(os.path.join("models", f))
            print(f"  {f}  ({size/1024/1024:.1f} MB)")
    else:
        print("  (empty)")

print(f"\n📂 base_model/")
print("-" * 40)
if os.path.exists("base_model"):
    items = os.listdir("base_model")
    if items:
        for f in sorted(items):
            size = os.path.getsize(os.path.join("base_model", f))
            print(f"  {f}  ({size/1024/1024:.1f} MB)")
    else:
        print("  (empty)")

## 3️⃣ Upload Files (Alternative)

If you prefer to upload manually instead of downloading from a link.

In [ ]:
# @title Upload Model File (.pth)
# @markdown Skip if you already downloaded a model above.
from google.colab import files
import shutil

if MODEL_PATH and os.path.exists(MODEL_PATH):
    print(f"✅ Model already set: {MODEL_PATH}")
    print("   (To re-upload, clear MODEL_PATH and re-run.)")
else:
    print("📁 Select your .pth model file:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        shutil.move(filename, f"models/{filename}")
        print(f"✅ Saved to models/{filename}")
    MODEL_PATH = f"models/{list(uploaded.keys())[0]}"
    print(f"\n📌 Model path: {MODEL_PATH}")

In [ ]:
# @title Upload Index File (.index)
# @markdown Skip if you already downloaded an index above.
from google.colab import files
import shutil

if INDEX_PATH and os.path.exists(INDEX_PATH):
    print(f"✅ Index already set: {INDEX_PATH}")
else:
    try:
        print("📁 Select your .index file (or cancel to skip):")
        uploaded = files.upload()
        for filename in uploaded.keys():
            shutil.move(filename, f"models/{filename}")
            print(f"✅ Saved to models/{filename}")
        INDEX_PATH = f"models/{list(uploaded.keys())[0]}"
        print(f"\n📌 Index path: {INDEX_PATH}")
    except:
        print("⏭️ Skipped — no index file loaded.")

In [ ]:
# @title Upload Audio to Convert
from google.colab import files
import shutil

print("📁 Select your audio file (.wav, .mp3, etc.):")
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, f"input/{filename}")
    print(f"✅ Saved to input/{filename}")
INPUT_PATH = f"input/{list(uploaded.keys())[0]}"
print(f"\n📌 Input path: {INPUT_PATH}")

## 4️⃣ Configure & Run Voice Conversion

Adjust the interactive form parameters below and run the conversion.

In [ ]:
# @title ⚙️ Conversion Settings

# @markdown ---
PITCH_SHIFT = 0          # @param {type:"slider", min:-12, max:12, step:1, description:"Semitones to shift pitch (+12 = 1 octave up)"}
PITCH_METHOD = "rmvpe"  # @param ["rmvpe", "harvest", "crepe", "pm", "dio", "crepe-tiny"] {allow-input: true}
INDEX_RATE = 0.5         # @param {type:"slider", min:0, max:1, step:0.05, description:"Search feature ratio (0 = disable)"}
FILTER_RADIUS = 3        # @param {type:"slider", min:0, max:7, step:1, description:"Median filter for pitch smoothing"}
PROTECT = 0.33           # @param {type:"slider", min:0, max:0.5, step:0.01, description:"Protect breath/consonant (0.33-0.5 recommended)"}
RMS_MIX_RATE = 1.0       # @param {type:"slider", min:0, max:1, step:0.05, description:"Envelope normalization mix rate"}
RESAMPLE_SR = 0          # @param {type:"slider", min:0, max:48000, step:1000, description:"Output sample rate (0 = keep original)"}
MODEL_VERSION = "v2"     # @param ["v1", "v2"]
# @markdown ---

print("⚙️  Conversion Settings:")
print(f"   Pitch shift:    {PITCH_SHIFT:+d} semitones")
print(f"   Pitch method:   {PITCH_METHOD}")
print(f"   Index rate:     {INDEX_RATE}")
print(f"   Filter radius:  {FILTER_RADIUS}")
print(f"   Protect:        {PROTECT}")
print(f"   RMS mix rate:   {RMS_MIX_RATE}")
print(f"   Resample SR:    {RESAMPLE_SR if RESAMPLE_SR > 0 else 'original'}")
print(f"   Model version:  {MODEL_VERSION}")

In [ ]:
# @title 🎤 Run Voice Conversion
import time
from rvc_python.infer import RVCInference

if not MODEL_PATH or not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}\n"
        "Please download or upload a .pth model file first."
    )

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"
print(f"🔧 Initializing RVC on {device}...")

t0 = time.time()
rvc = RVCInference(device=device)
print(f"✅ RVC initialized in {time.time() - t0:.1f}s")

print(f"\n📦 Loading model: {MODEL_PATH}")
t1 = time.time()
index = INDEX_PATH if INDEX_PATH and os.path.exists(INDEX_PATH) else ""
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=index)
print(f"✅ Model loaded in {time.time() - t1:.1f}s")

rvc.set_params(
    f0up_key=PITCH_SHIFT,
    f0method=PITCH_METHOD,
    index_rate=INDEX_RATE,
    filter_radius=FILTER_RADIUS,
    protect=PROTECT,
    rms_mix_rate=RMS_MIX_RATE,
    resample_sr=RESAMPLE_SR,
)

input_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]
output_path = f"output/{input_name}_converted.wav"

print(f"\n🎤 Converting: {INPUT_PATH}")
print(f"   Output:   {output_path}")
t2 = time.time()
rvc.infer_file(INPUT_PATH, output_path)
elapsed = time.time() - t2
print(f"✅ Conversion complete in {elapsed:.1f}s")

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"📁 Output: {output_path} ({size_mb:.1f} MB)")

## 5️⃣ Listen & Download

In [ ]:
# @title 🔊 Play Converted Audio
from IPython.display import Audio, display

print("🔊 Converted audio:")
display(Audio(output_path))

In [ ]:
# @title 🔊 Play Original Audio (Comparison)
from IPython.display import Audio, display

print("🔊 Original audio:")
display(Audio(INPUT_PATH))

In [ ]:
# @title 💾 Download Converted Audio
from google.colab import files

print("💾 Downloading...")
files.download(output_path)

## 6️⃣ Batch Conversion

Upload multiple audio files to `input/` and convert them all at once.

In [ ]:
# @title Upload Multiple Audio Files
from google.colab import files
import shutil

print("📁 Select multiple audio files:")
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, f"input/{filename}")
    print(f"✅ input/{filename}")
print(f"\n📂 {len(uploaded)} file(s) uploaded")

In [ ]:
# @title 🔄 Run Batch Conversion
import time
from rvc_python.infer import RVCInference

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"

rvc = RVCInference(device=device)
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=index)
rvc.set_params(
    f0up_key=PITCH_SHIFT,
    f0method=PITCH_METHOD,
    index_rate=INDEX_RATE,
    filter_radius=FILTER_RADIUS,
    protect=PROTECT,
    rms_mix_rate=RMS_MIX_RATE,
    resample_sr=RESAMPLE_SR,
)

t0 = time.time()
results = rvc.infer_dir("input", "output")
elapsed = time.time() - t0

print(f"✅ Converted {len(results)} files in {elapsed:.1f}s")
for r in results:
    size_mb = os.path.getsize(r) / 1024 / 1024
    print(f"  📁 {r} ({size_mb:.1f} MB)")

In [ ]:
# @title 📦 Download All as ZIP
import zipfile
from google.colab import files

zip_path = "converted_audio.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk("output"):
        for file in files_list:
            if file.endswith('.wav'):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, "output")
                zf.write(file_path, arcname)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"📦 {zip_path} ({size_mb:.1f} MB)")
files.download(zip_path)

## 7️⃣ A/B Pitch Experiment

Compare the same audio with different pitch shifts.

In [ ]:
# @title 🎵 A/B Pitch Comparison
# @markdown Select pitch values to compare side by side.
PITCH_A = -5       # @param {type:"slider", min:-12, max:12, step:1}
PITCH_B = 0        # @param {type:"slider", min:-12, max:12, step:1}
PITCH_C = 3        # @param {type:"slider", min:-12, max:12, step:1}
PITCH_D = 7        # @param {type:"slider", min:-12, max:12, step:1}

from IPython.display import Audio, display
from rvc_python.infer import RVCInference
import time

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"
rvc = RVCInference(device=device)
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=index)

input_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]

for shift in [PITCH_A, PITCH_B, PITCH_C, PITCH_D]:
    rvc.set_params(f0up_key=shift, f0method=PITCH_METHOD)
    out = f"output/{input_name}_pitch{shift:+d}.wav"
    t0 = time.time()
    rvc.infer_file(INPUT_PATH, out)
    elapsed = time.time() - t0
    print(f"🎵 Pitch {shift:+d} semitones ({elapsed:.1f}s):")
    display(Audio(out))
    print()

---

## Tips

| Parameter | Recommendation |
|-----------|---------------|
| **Pitch method** | `rmvpe` for best quality. `crepe` better for clean speech but slower. |
| **Index rate** | `0.5-0.8` with index file. `0` to ignore index. |
| **Protect** | `0.33-0.5` to preserve breath/consonants. |
| **Pitch shift** | `+12` = octave up, `-12` = octave down. |
| **GPU memory** | OOM? Restart runtime and use CPU mode. |

## Troubleshooting

- **CUDA OOM**: Smaller model or switch to CPU
- **Bad quality**: Try different pitch method or adjust index rate
- **Wrong voice**: Check your `.pth` model file
- **Install errors**: Ensure GPU runtime, re-run install cell
- **Download failed**: Check link, try alternate source, or upload manually
- **GDrive quota**: Large files may hit limits, try `gdown` or direct link

## Links

- **GitHub**: [onxlmao/rvc-python](https://github.com/onxlmao/rvc-python)
- **Issues**: [Report a bug](https://github.com/onxlmao/rvc-python/issues)